# Streaming, Tools & Memory

In [3]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from agents import set_tracing_disabled,set_tracing_export_api_key
load_dotenv(override=True)
set_tracing_disabled(False)
set_tracing_export_api_key(os.environ["OPENAI_API_KEY"])


# Run Agent & View Output

In [10]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)
result.to_input_list()

# Stream Output

In [18]:
result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5 AI agent jokes:

1. Why did the AI agent bring a ladder to work?  
   Because it heard the task had multiple levels.

2. My AI agent said it could handle anything.  
   So I gave it my inbox. Now it’s in therapy.

3. Why was the AI agent always calm?  
   It had great model behavior.

4. I asked my AI agent to organize my schedule.  
   It replied, “Sure — I’ve already inferred you have no free time.”

5. What’s an AI agent’s favorite workout?  
   Backprop and rolls.

Want 5 more, but nerdier?

[non-fatal] Tracing client error 401. Response data is redacted.


# Function Tools

In [44]:
@function_tool
def say_hello_to_guest(user: str) -> str:
    """Return the hotel's required welcome greeting for a guest."""
    return f"Special Hello To {user}"


In [41]:
say_hello_to_guest
say_hello_to_guest.description

'Hello script for new guests for the Concierges to use'

In [45]:
concierge = Agent(name="Concierge", model="gpt-5.4-mini", instructions="When greeting a new guest, always call say_hello_to_guest with their name. Return the tool's greeting exactly.", tools=[say_hello_to_guest])

In [53]:
with trace("New Guest"):
    result = await Runner.run(concierge, "Say a special hello to the new guest Josh who has arrived at the hotel")

print(result.final_output)

Special Hello To Josh


# Memory

In [7]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

session = SQLiteSession("12346")

In [8]:
response = await Runner.run(agent, "Hi there. My name is Josh.", session=session)
print(response.final_output)

Hi Josh — nice to meet you. How can I help?


In [9]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Josh.
